In [1]:
import pandas as pd


In [ ]:
learn_df = pd.read_csv("modal_experiment_v1_training_positive_flavour_pilot.csv"
                       ).drop(columns=["index", "index.1"])

# throw out wh-question and individual scenarios that performed worst during norming
to_keep = ["T3","T5","T7","T9","T11","T12"]
learn_df = learn_df[learn_df["item_id"].isin(to_keep)].reset_index(drop=True)

# change instances of 'mip' to 'dax'
learn_df["lexical_adjective_target"] = learn_df["lexical_adjective_target"].apply(
    lambda x: x.replace("mip", "dax")
)

learn_df.head()


,phase,item_id,domain,training_flavor,syntax,D_status,E_status,temporal_orientation,order_pair_id,order_1,...,expected_context_E,expected_whole_scenario_deontic_interpretation,expected_whole_scenario_epistemic_interpretation,notes,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,comprehension_type,comprehension_keep
0,Training,T3,Community center,Root/deontic,Declarative,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Pilot rewrite: multiple available alternatives...,Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,A,diagnostic_strong,True
1,Training,T3,Community center,Epistemic,Declarative,D unspecified,E+ (weak / indirect evidence),past perfect / present progressive or stative,TE,N-E,...,E+,Expected non-deontic interpretation,Expected epistemic interpretation,Pilot rewrite: multiple available alternatives...,Where did Nora's friend see her heading?,Toward the swimming and climbing areas,Toward the fitness area,A,diagnostic_strong,True
2,Training,T5,Parking,Root/deontic,Yes/no inversion,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Pilot rewrite: multiple available alternatives...,What kind of event is Omar attending?,A museum exhibition,A concert,B,neutral,False
3,Training,T5,Parking,Epistemic,Yes/no inversion,D unspecified,E+ (weak / indirect evidence),past perfect / present progressive or stative,TE,N-E,...,E+,Expected non-deontic interpretation,Expected epistemic interpretation,Pilot rewrite: multiple available alternatives...,What kind of event did Omar attend?,A museum exhibition,A concert,B,neutral,False
4,Training,T7,Art studio,Root/deontic,Declarative,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Pilot rewrite: multiple available alternatives...,What enables Eli to use the print room?,His advanced-course qualification,A visitor wristband,A,diagnostic_weak,True


In [3]:
learn_df["comprehension_keep"] = (learn_df["comprehension_type"].isin(["diagnostic_strong", "diagnostic_moderate", "diagnostic_weak"]))
print(learn_df["comprehension_keep"].tolist())

learn_df["comprehension_question"] = learn_df.apply(
    lambda x: x["comprehension_question"] if x["comprehension_keep"] else "",
    axis=1
)
learn_df["comprehension_question"].to_list()


[True, True, False, False, True, True, True, True, False, False]


["Which area does Nora's membership allow her to use?",
 "Where did Nora's friend see her heading?",
 '',
 '',
 'What enables Eli to use the print room?',
 'Which two rooms might Eli have worked in?',
 'What does Mateo have for entering the festival?',
 'Which entrance is explicitly mentioned as one possibility?',
 '',
 '']

In [4]:
print(learn_df["E_status"].value_counts())

print(learn_df["D_status"].value_counts())


E_status
E unspecified / not ruled out    5
E+ (weak / indirect evidence)    5
Name: count, dtype: int64
D_status
D+               5
D unspecified    5
Name: count, dtype: int64


In [5]:
learn_df["D_status"] = learn_df["D_status"].map({"D+": 1, "D unspecified": 0, "D-": -1})
learn_df["E_status"] = learn_df["E_status"].map({"E+ (weak / indirect evidence)": 1, "E unspecified / not ruled out": 0})

print(learn_df["D_status"].tolist())
print(learn_df["E_status"].tolist())


[1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
[0, 1, 0, 1, 0, 1, 0, 1, 0, 1]


In [6]:
order1_df = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_1",
            "context_order_1",
            "functional_auxiliary_target",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "order_1":"order",
                "context_order_1":"context_order"
                })
order1_df["order_group"] = (order1_df.reset_index().index % 4) < 2

order2_df = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_2",
            "context_order_2",
            "functional_auxiliary_target",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"]].rename(columns={
                "order_2":"order",
                "context_order_2":"context_order"
                })
order2_df["order_group"] = (order1_df.reset_index().index % 4) >= 2
learn_df = pd.concat([order1_df, order2_df])
learn_df["order_group"] = learn_df["order_group"].apply(int).apply(str)

print(order1_df["order_group"].tolist())
print(order2_df["order_group"].tolist())

print(learn_df["order"].tolist())
learn_df.groupby(["order_group", "training_flavor"])["order"].value_counts()


[True, True, False, False, True, True, False, False, True, True]
[False, False, True, True, False, False, True, True, False, False]
['N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N']


order_group  training_flavor  order
0            Epistemic        E-N      3
                              N-E      2
             Root/deontic     D-N      3
                              N-D      2
1            Epistemic        N-E      3
                              E-N      2
             Root/deontic     N-D      3
                              D-N      2
Name: count, dtype: int64

In [7]:

functional = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_group",
            "order",
            "context_order",
            "functional_auxiliary_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "functional_auxiliary_target":"alien_target_response",
                })
functional["modal_type"] = "functional"

lexical = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_group",
            "order",
            "context_order",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "lexical_adjective_target":"alien_target_response",
                })
lexical["modal_type"] = "lexical"

learn_df = pd.concat([functional, lexical])
learn_df.head()


,phase,item_id,domain,training_flavor,D_status,E_status,temporal_orientation,order_group,order,context_order,alien_target_response,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,modal_type
0,Training,T3,Community center,Root/deontic,1,0,future eventive,1,N-D,Nora is going to the community center this aft...,"Oh, so nora dax use the pool this afternoon.",Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,A,functional
1,Training,T3,Community center,Epistemic,0,1,past perfect / present progressive or stative,1,N-E,Nora is at the community center right now. A f...,"Oh, so nora dax be swimming now.",Where did Nora's friend see her heading?,Toward the swimming and climbing areas,Toward the fitness area,A,functional
2,Training,T5,Parking,Root/deontic,1,0,future eventive,0,N-D,Omar is driving to a concert tomorrow. His par...,"Oh, so dax Omar park in the west lot tomorrow?...",,A museum exhibition,A concert,B,functional
3,Training,T5,Parking,Epistemic,0,1,past perfect / present progressive or stative,0,N-E,Omar drove to a concert earlier today. His fri...,"Oh, so dax Omar be parked in the west lot now?...",,A museum exhibition,A concert,B,functional
4,Training,T7,Art studio,Root/deontic,1,0,future eventive,1,N-D,Eli is taking an art workshop tomorrow. His ad...,"Oh, so eli dax use the print room tomorrow.",What enables Eli to use the print room?,His advanced-course qualification,A visitor wristband,A,functional


In [8]:
def fix_name(x):
    s = x.split(" ")
    name = s[2]
    if name == "it" or name == "is" or name == "dax":
        return x
    print(name)
    s[2] = name.capitalize()
    x = " ".join(s)
    print(x)
    return x

# fix uncapitalized names
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(fix_name)


nora
Oh, so Nora dax use the pool this afternoon.
nora
Oh, so Nora dax be swimming now.
eli
Oh, so Eli dax use the print room tomorrow.
eli
Oh, so Eli dax have worked in the print room this morning.
daniel
Oh, so Daniel dax work in Lab B tomorrow.
daniel
Oh, so Daniel dax be in Lab B now.
nora
Oh, so Nora dax use the pool this afternoon.
nora
Oh, so Nora dax be swimming now.
eli
Oh, so Eli dax use the print room tomorrow.
eli
Oh, so Eli dax have worked in the print room this morning.
daniel
Oh, so Daniel dax work in Lab B tomorrow.
daniel
Oh, so Daniel dax be in Lab B now.


In [9]:
learn_df["comprehension_correct"] = learn_df.apply(
    lambda x: x["comprehension_option_A"] if x["comprehension_correct"] == "A" else x["comprehension_option_B"],
    axis=1
)

learn_df["question_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.split(" — ")[1] if " — " in x else ""
)
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.split(" — ")[0]
)
# make dax bolded
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.replace("dax", "<b>dax</b>")
)

learn_df["group"] = learn_df.apply(
    lambda x: "D->E_" + x["modal_type"] if x["training_flavor"] == "Root/deontic" else "E->D_" + x["modal_type"],
    axis=1
) + "_" + learn_df["order_group"]


learn_df.to_csv("exp1_learning_stimulus.csv")
learn_df.head()


,phase,item_id,domain,training_flavor,D_status,E_status,temporal_orientation,order_group,order,context_order,alien_target_response,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,modal_type,question_response,group
0,Training,T3,Community center,Root/deontic,1,0,future eventive,1,N-D,Nora is going to the community center this aft...,"Oh, so Nora <b>dax</b> use the pool this after...",Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,The swimming pool,functional,,D->E_functional_1
1,Training,T3,Community center,Epistemic,0,1,past perfect / present progressive or stative,1,N-E,Nora is at the community center right now. A f...,"Oh, so Nora <b>dax</b> be swimming now.",Where did Nora's friend see her heading?,Toward the swimming and climbing areas,Toward the fitness area,Toward the swimming and climbing areas,functional,,E->D_functional_1
2,Training,T5,Parking,Root/deontic,1,0,future eventive,0,N-D,Omar is driving to a concert tomorrow. His par...,"Oh, so <b>dax</b> Omar park in the west lot to...",,A museum exhibition,A concert,A concert,functional,Yes.,D->E_functional_0
3,Training,T5,Parking,Epistemic,0,1,past perfect / present progressive or stative,0,N-E,Omar drove to a concert earlier today. His fri...,"Oh, so <b>dax</b> Omar be parked in the west l...",,A museum exhibition,A concert,A concert,functional,Yes.,E->D_functional_0
4,Training,T7,Art studio,Root/deontic,1,0,future eventive,1,N-D,Eli is taking an art workshop tomorrow. His ad...,"Oh, so Eli <b>dax</b> use the print room tomor...",What enables Eli to use the print room?,His advanced-course qualification,A visitor wristband,His advanced-course qualification,functional,,D->E_functional_1


In [10]:
learn_df.groupby(["group", "training_flavor", "modal_type"])["order"].value_counts()


group              training_flavor  modal_type  order
D->E_functional_0  Root/deontic     functional  D-N      3
                                                N-D      2
D->E_functional_1  Root/deontic     functional  N-D      3
                                                D-N      2
D->E_lexical_0     Root/deontic     lexical     D-N      3
                                                N-D      2
D->E_lexical_1     Root/deontic     lexical     N-D      3
                                                D-N      2
E->D_functional_0  Epistemic        functional  E-N      3
                                                N-E      2
E->D_functional_1  Epistemic        functional  N-E      3
                                                E-N      2
E->D_lexical_0     Epistemic        lexical     E-N      3
                                                N-E      2
E->D_lexical_1     Epistemic        lexical     N-E      3
                                                E-N      2
Na

In [11]:
pilot_condition = pd.read_csv("modal_experiment_v1_positive_flavour_test_pilot.csv").drop(columns=["index", "Unnamed: 0"])
print(pilot_condition["group"].value_counts())

# change instances of 'mip' to 'dax'
pilot_condition["alien_target_response"] = pilot_condition["alien_target_response"].apply(
    lambda x: x.replace("mip", "dax")
)
# make dax bolded
pilot_condition["alien_target_response"] = pilot_condition["alien_target_response"].apply(
    lambda x: x.replace("dax", "<b>dax</b>")
)

# Existing exclusions retained; revise these after the new norming results.
to_drop = ["F1", "F6", "C1", "C2", "C4", "C5"]
pilot_condition = pilot_condition[~pilot_condition["item_id"].isin(to_drop)].reset_index(drop=True)

pilot_condition["group"] = pilot_condition["group"].apply(
    lambda x: x.split("_flavour")[0]
)

# fix uncapitalized names
pilot_condition["alien_target_response"] = pilot_condition["alien_target_response"].apply(fix_name)

pilot_condition.head()


group
D->E_functional_0    12
D->E_functional_1    12
E->D_functional_0    12
E->D_functional_1    12
D->E_lexical_0       12
D->E_lexical_1       12
E->D_lexical_0       12
E->D_lexical_1       12
Name: count, dtype: int64
grace
Oh, so Grace <b>dax</b> have gone into the sound-mixing suite.
marcus
Oh, so Marcus <b>dax</b> have waited in the premium lounge.
elena
Oh, so Elena <b>dax</b> be sitting in the live recording booth now.
ravi
Oh, so Ravi <b>dax</b> have gone into the equipment storage room.
zoe
Oh, so Zoe <b>dax</b> be working at the decorating station now.
henry
Oh, so Henry <b>dax</b> practice in the faculty room tomorrow afternoon.
leah
Oh, so Leah <b>dax</b> go into the equipment bay tomorrow morning.
colin
Oh, so Colin <b>dax</b> have gone into the loading bay.
grace
Oh, so Grace <b>dax</b> have gone into the sound-mixing suite.
marcus
Oh, so Marcus <b>dax</b> have waited in the premium lounge.
elena
Oh, so Elena <b>dax</b> be sitting in the live recording booth now.
ravi

,phase,item_id,domain,condition,direction,direction_or_condition,D_status,E_status,P_status,Nec_status,temporal_orientation,order_group,order,context_order,expected_choice,alien_target_response,modal_type,learning_group,group
0,Critical Test,C3,Student film studio,flavour_positive,D->E,Root→Epistemic,NaN,1.0,NaN,NaN,past perfect,0,N-E-D,Grace is working on a documentary at a student...,Accept,"Oh, so Grace <b>dax</b> have gone into the sou...",functional,D->E_functional_0,D->E_functional_0
1,Critical Test,C6,Airport premium lounge,flavour_positive,D->E,Root→Epistemic,NaN,1.0,NaN,NaN,past perfect,0,N-E-D,Marcus waited several hours at the airport yes...,Accept,"Oh, so Marcus <b>dax</b> have waited in the pr...",functional,D->E_functional_0,D->E_functional_0
2,Critical Test,C7,College radio station,flavour_positive,D->E,Root→Epistemic,NaN,1.0,NaN,NaN,present progressive,0,N-E-D,Elena is visiting a college radio station. Her...,Accept,"Oh, so Elena <b>dax</b> be sitting in the live...",functional,D->E_functional_0,D->E_functional_0
3,Critical Test,C8,Ski lodge equipment storage,flavour_positive,D->E,Root→Epistemic,NaN,1.0,NaN,NaN,past perfect,0,N-E-D,Ravi spent the morning at a ski lodge. His fri...,Accept,"Oh, so Ravi <b>dax</b> have gone into the equi...",functional,D->E_functional_0,D->E_functional_0
4,Filler,F3,Bakery decorating station,flavour_positive,D->E,Root→Epistemic,1.0,1.0,NaN,NaN,present progressive,0,N-E-D,Zoe is helping at a bakery fundraiser this aft...,Accept,"Oh, so Zoe <b>dax</b> be working at the decora...",functional,D->E_functional_0,D->E_functional_0


In [12]:
pilot_condition.to_csv("exp1_test_stimulus_pilot.csv")


Participant assignment: choose one `group` from `exp1_test_stimulus.csv` and show only test rows matching it. Use that group's `learning_group` to select rows from the unchanged `exp1_learning_stimulus.csv`. For example, `D->E_functional_0_force_negative` uses learning group `D->E_functional_0`. There are 24 testing groups: 3 conditions × 2 learned flavours × 2 modal types × 2 order groups. The legacy `direction` label identifies the learning group; in force-negative testing, the flavour remains the learned flavour.

The existing `to_keep` and `to_throw` lists are retained for later revision. Testing uses only Critical Test and Filler rows from the three supplied sets. Fillers remain shared across the two learning directions, as in the original code. No learning cell source has changed. The separate training CSV referenced in cell 1 is still required to run the full notebook.


`expected_choice` is `Accept` or `Reject` for the displayed target, accounting for outer negation. For criticals this assumes successful transfer (to the other flavour, or from possibility to necessity); all current criticals predict `Accept`. Filler readings agree and give the same expected choice with or without transfer. Labels describe endorsement, not button positions or numeric response codes. Expected choices are derived from the status columns, not item IDs.